# 6. Datakvalitet, felkällor och bias

Förra notebooken slutade med en brasklapp: allt förutsatte att stickprovet var **slumpmässigt**. Då är felet slumpmässigt,
det krymper när stickprovet växer och vi kan räkna ut hur stort det brukar vara.

Verkliga undersökningar har fler problem än så. Människor svarar inte, register är inaktuella och frågor missförstås.
Sådana fel kan vara **systematiska**, och då hjälper det inte att samla in mer data.

**Efter den här notebooken kan du**
- skilja på slumpmässiga fel och systematiska fel (bias)
- beskriva felkällorna urvalsfel, täckningsfel, bortfall och mätfel
- förklara varför ett stort stickprov inte botar bias
- resonera om åt vilket håll en undersökning kan vara snedvriden

**Förkunskaper:** [5. Population och stickprov](../05_population_och_stickprov/05_population_och_stickprov.ipynb)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

N = 10_000
population = pd.DataFrame({
    "height": rng.normal(172, 9, N).round(1),                       # cm
    "num_children": rng.poisson(1.5, N),
    "education": rng.choice(["Grundskola", "Gymnasium", "Högskola"], N, p=[0.2, 0.45, 0.35]),
    "income": rng.lognormal(np.log(32), 0.4, N).round(1),           # tkr per månad
})
true_mean_income = population["income"].mean()
print(f"Sann medelinkomst i populationen: {true_mean_income:.1f} tkr per månad")

## 6.1 Felkällor i en undersökning

| Felkälla | Vad det är | Exempel |
|---|---|---|
| **Urvalsfel** | Slumpmässig skillnad mellan stickprov och population | Minskar med större *n* |
| **Övertäckning** | Urvalsramen innehåller individer som inte hör till populationen | Register med personer som har flyttat |
| **Undertäckning** | Individer i populationen saknas i urvalsramen | Webbenkät når inte dem som saknar internet |
| **Bortfall** | Utvalda individer svarar inte | Låg svarsfrekvens i enkäter |
| **Mätfel** | Det registrerade värdet skiljer sig från det sanna | Ledande frågor, eller att man uppger lägre vikt än man har |

**Urvalsramen** är den förteckning som stickprovet dras från, till exempel ett kundregister.

Bara den första felkällan är rent slumpmässig. De övriga kan ge **bias**: ett systematiskt fel som drar resultatet åt
ett bestämt håll. Ett stickprov med bias är inte **representativt** för populationen.

## 6.2 Bortfall: när de som svarar inte liknar de som inte svarar

Vi skickar en inkomstenkät till *alla* 10 000 i populationen. Anta att personer med hög inkomst är mindre benägna att svara.
Parametern `strength` styr hur starkt svarsviljan beror på inkomsten.

🔮 **Gissa först:** vi får flera tusen svar. Blir medelinkomsten bland de svarande för hög, för låg eller ungefär rätt?

In [ ]:
strength = 1.0        # ← ändra: 0 = bortfallet beror inte på inkomst, 2 = starkt beroende

response_prob = np.clip(0.56 - strength * (population["income"] - 32) / 50, 0.05, 0.95)
responds = rng.random(N) < response_prob
respondents = population[responds]

print(f"Antal svar: {len(respondents)}  (svarsfrekvens {responds.mean():.0%})")
print(f"Sann medelinkomst:            {true_mean_income:.1f} tkr")
print(f"Medelinkomst bland svarande:  {respondents['income'].mean():.1f} tkr")
print(f"Fel:                          {respondents['income'].mean() - true_mean_income:+.1f} tkr")

🧪 **Labba:**
1. Sätt `strength = 0`. Svarsfrekvensen är fortfarande bara drygt hälften – men vad händer med felet?
2. Pröva `strength = 2`. Vad händer?
3. Pröva `strength = -1`. Vilka är det nu som svarar oftare, och åt vilket håll går felet?

<details>
<summary>💡 <b>Vad visar labben?</b></summary>

Med `strength = 0` är bortfallet **slumpmässigt**: de som svarar liknar dem som inte svarar, och skattningen blir bra trots
att nästan hälften saknas. Det är inte bortfallets *storlek* som är problemet, utan att det hänger ihop med det vi mäter.
Ju starkare kopplingen är, desto större bias. Med negativ `strength` svarar höginkomsttagare oftare, och felet byter riktning.

</details>

I verkligheten ser vi bara svaren. Vi kan inte räkna ut felet, eftersom vi inte vet vad de som inte svarade skulle ha sagt.

## 6.3 Stort och snett eller litet och slumpmässigt?

Kör om enkäten med `strength = 1.0` och jämför med ett litet OSU på 200 personer som alla svarar.

In [ ]:
response_prob = np.clip(0.56 - 1.0 * (population["income"] - 32) / 50, 0.05, 0.95)
respondents = population[rng.random(N) < response_prob]
small_srs = population["income"].sample(200, random_state=2)

print(f"Sanningen:                              {true_mean_income:.1f} tkr")
print(f"Enkät med bortfall, n = {len(respondents)}:           {respondents['income'].mean():.1f} tkr")
print(f"OSU utan bortfall, n = 200:             {small_srs.mean():.1f} tkr")

Ett enda försök kan vara tur. Vi upprepar båda undersökningarna 500 gånger. För att jämförelsen ska bli rättvis låter vi
båda ha lika många svar den här gången, 200 stycken:

In [ ]:
incomes = population["income"].to_numpy()
prob = response_prob.to_numpy()

srs_means, biased_means = [], []
for _ in range(500):
    srs_means.append(rng.choice(incomes, 200, replace=False).mean())
    answered = incomes[rng.random(N) < prob]                # de som skulle svara
    biased_means.append(rng.choice(answered, 200, replace=False).mean())

plt.hist(srs_means, bins=25, alpha=0.7, label="OSU utan bortfall")
plt.hist(biased_means, bins=25, alpha=0.7, label="Systematiskt bortfall")
plt.axvline(true_mean_income, color="red", linestyle="--", label="sanningen")
plt.xlabel("Skattad medelinkomst (tkr per månad)")
plt.ylabel("Antal undersökningar")
plt.legend()
plt.show()

Båda histogrammen är smala (det snedvridna är till och med något smalare). Men det ena är centrerat kring sanningen och det andra kring fel värde.

- **Slumpmässigt fel** = spridningen kring mitten. Syns som histogrammets bredd.
- **Systematiskt fel (bias)** = hela histogrammet ligger fel.

Tänk på en piltavla: slumpfel är pilar som sprider sig runt mitten, bias är ett sikte som är felinställt.

🔮 **Gissa först:** vad händer med de två histogrammen om vi ökar från 200 till 2 000 svar per undersökning?

In [ ]:
num_answers = 200        # ← ändra: pröva 50, 200 och 2000

srs_means, biased_means = [], []
for _ in range(500):
    srs_means.append(rng.choice(incomes, num_answers, replace=False).mean())
    answered = incomes[rng.random(N) < prob]
    biased_means.append(rng.choice(answered, num_answers, replace=False).mean())

plt.hist(srs_means, bins=25, alpha=0.7, label="OSU utan bortfall")
plt.hist(biased_means, bins=25, alpha=0.7, label="Systematiskt bortfall")
plt.axvline(true_mean_income, color="red", linestyle="--", label="sanningen")
plt.xlim(24, 42)
plt.xlabel("Skattad medelinkomst (tkr per månad)")
plt.ylabel("Antal undersökningar")
plt.title(f"{num_answers} svar per undersökning")
plt.legend()
plt.show()

print(f"OSU:      medel {np.mean(srs_means):.1f}, spridning {np.std(srs_means):.2f}")
print(f"Bortfall: medel {np.mean(biased_means):.1f}, spridning {np.std(biased_means):.2f}")

🧪 **Labba:** Pröva 50, 200 och 2 000 svar.

> **Den viktigaste lärdomen i den här notebooken:** ett större stickprov minskar det *slumpmässiga* felet men gör
> ingenting åt *bias*. Med 2 000 svar blir den snedvridna undersökningen bara mer exakt fel – ett smalt histogram
> långt från sanningen. Stort n kan till och med vara farligt, eftersom det ger ett falskt intryck av säkerhet.

## 6.4 Undertäckning: när urvalsramen inte når alla

Ett företag vill veta medelinkomsten och drar ett korrekt OSU – men ur sitt **kundregister**. Anta att sannolikheten
att vara kund ökar med inkomsten.

In [ ]:
customer_prob = np.clip(population["income"] / 80, 0, 1)
is_customer = rng.random(N) < customer_prob
frame = population[is_customer]                       # urvalsramen: bara kunderna

frame_sample = frame["income"].sample(500, random_state=3)
print(f"Andel av populationen som finns i urvalsramen: {is_customer.mean():.0%}")
print(f"Sann medelinkomst:               {true_mean_income:.1f} tkr")
print(f"OSU ur kundregistret (n = 500):  {frame_sample.mean():.1f} tkr")

Urvalet var helt slumpmässigt – men ur fel grupp. Slumpen kan bara ge ett representativt stickprov av det som finns i urvalsramen.

✏️ **Din tur:** Kundregistret innehåller färre låginkomsttagare. Undersök hur det påverkar en *annan* variabel:
jämför medellängden (`height`) i `frame` med medellängden i `population`. Blir det någon bias? Varför, eller varför inte?

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
print(f"Medellängd i populationen:  {population['height'].mean():.2f} cm")
print(f"Medellängd i urvalsramen:   {frame['height'].mean():.2f} cm")
# Nästan ingen skillnad. I den här populationen hänger längd inte ihop med inkomst,
# så en urvalsram som är sned i inkomst blir inte sned i längd.
```

</details>

<details>
<summary>💡 <b>Slutsats av övningen</b></summary>

Täckningsfel och bortfall ger bara bias för variabler som **hänger ihop med** orsaken till att man saknas.
Samma snedvridna urvalsram kan alltså ge bra skattningar av en variabel och usla skattningar av en annan.
Frågan att ställa är alltid: *skiljer sig de som saknas från de som finns med, i just det jag mäter?*

</details>

## 6.5 Mätfel: när svaren inte är sanna

Även med ett perfekt urval kan själva mätningen vara fel. Anta att folk i en enkät rundar av sin längd uppåt,
i genomsnitt med 1,5 cm, och dessutom minns lite fel (slumpmässigt, ±1 cm).

In [ ]:
sample = population.sample(1000, random_state=4)

systematic_error = 1.5      # ← ändra: pröva 0
random_error = 1.0          # ← ändra: pröva 5

reported = sample["height"] + systematic_error + rng.normal(0, random_error, len(sample))

print(f"Sann medellängd i stickprovet:   {sample['height'].mean():.2f} cm")
print(f"Medel av uppgiven längd:         {reported.mean():.2f} cm")

🧪 **Labba:**
1. Sätt `systematic_error = 0` och `random_error = 5`: stora men slumpmässiga mätfel. Hur mycket påverkas medelvärdet?
2. Sätt tillbaka `systematic_error = 1.5` och `random_error = 0`. Vad händer?

Slumpmässiga mätfel tar till stor del ut varandra i ett medelvärde. Systematiska mätfel gör det inte.

## 6.6 Tänk som en granskare

✏️ **Din tur:** Ingen kod här. Fundera på varje fall innan du fäller ut svaret:
*vilken felkälla, och åt vilket håll bör resultatet vara snedvridet?*

**Fall A.** En tidning lägger ut en webbenkät: "Hur många timmar per dag använder du internet?" och får 50 000 svar.

**Fall B.** Ett gym mejlar alla medlemmar och frågar hur ofta de tränar. 15 % svarar.

**Fall C.** En forskare frågar: "Håller du med om att den onödigt höga bensinskatten bör sänkas?"

**Fall D.** En kommun ringer upp ett slumpmässigt urval invånare på fasta telefoner, vardagar klockan 10–15.

<details>
<summary>💡 <b>Svar</b></summary>

**A.** Undertäckning och självselektion: bara den som är på nätet och besöker tidningens sajt kan svara, och den som inte
använder internet kan inte komma med alls. Resultatet överskattar internetanvändningen. De 50 000 svaren hjälper inte.

**B.** Bortfall, troligen systematiskt: den som tränar ofta är mer intresserad av att svara, och den som slutat gå dit
skäms kanske. Dessutom mätfel, eftersom folk gärna överskattar sin träning. Resultatet blir för högt.

**C.** Mätfel genom en ledande fråga. "Onödigt höga" styr mot att hålla med. Andelen som vill sänka skatten överskattas.

**D.** Undertäckning (många saknar fast telefon, särskilt yngre) och bortfall (de som arbetar dagtid svarar inte).
De som nås är till stor del äldre och hemmavarande. Riktningen på felet beror på vad som frågas, men stickprovet är inte representativt.

</details>

## Sammanfattning

- **Slumpmässiga fel** tar ut varandra i längden och minskar med större stickprov.
- **Systematiska fel (bias)** drar åt ett håll och påverkas inte av stickprovets storlek.
- Bortfall och täckningsfel ger bias när de som saknas **skiljer sig** från de som finns med, i det som mäts.
- Ett litet slumpmässigt stickprov är ofta bättre än ett stort snedvridet.
- Konfidensintervall och tester, som kommer härnäst, mäter **bara det slumpmässiga felet**. De säger ingenting om bias.

### Kontrollfrågor
1. En enkät har 90 % bortfall. Är resultatet säkert snedvridet?
2. Varför kan man inte "räkna bort" bortfallsbias i efterhand genom att titta på svaren?

<details>
<summary>💡 <b>Svar</b></summary>

1. Nej, inte säkert. Om bortfallet inte hänger ihop med det som mäts blir det ingen bias. Men med så stort bortfall är
   risken stor, och det går sällan att kontrollera.
2. För att biasen beror på vad de som *inte* svarade skulle ha sagt, och den informationen finns inte i datan.

</details>

**Nästa notebook:** [7. Punktskattning och konfidensintervall](../07_konfidensintervall/07_konfidensintervall.ipynb)